In [1]:
# we will need the credentials we saved in the .env file
from dotenv import dotenv_values

# We also will need SQLAlchemy and its functions
from sqlalchemy import create_engine, types
from sqlalchemy.dialects.postgresql import JSON as postgres_json

import pandas as pd

# requests library will make the API calls. 
# the json package will parse the JSON string and convert it to Python data structures
import requests
import json

# with 'datetime' we want to catch the timestamp of the API call. For the actuality reference. 
# and 'time' for slowing down a .bit
from datetime import datetime
import time

In [3]:
airport_staids = {
    'LAX': 72295 #Los Angeles International Airport
    ,'JFK': 71845 #John F. Kennedy International Airport New York
    ,'DEN': 72565 # Denver International Airport
    ,'DTW': 72537 # Detroit Metropolitan airport
    ,'ORD': 72530 #Chicago 
    ,'ATL': 72219 #Atlanta    
     }

In [5]:
# getting API and DB credentials - Alternative 1: dotenv_values()

config = dotenv_values()

api_key = config['x-rapidapi-key'] # align the key label with your .env file

In [19]:
first_days=pd.date_range(start='11/12/2022', end='31/01/2023', freq='MS')
last_days = first_days + pd.offsets.MonthEnd() # see, what we did here? DRY rules! :)
first_days_list = first_days.strftime('%Y-%m-%d').tolist()
last_days_list = last_days.astype(str).tolist()
print(first_days_list) 
print(last_days_list)
monthly_ranges = [(start_date, end_date) for start_date, end_date in zip(first_days_list, last_days_list)]

['2022-12-01', '2023-01-01']
['2022-12-31', '2023-01-31']


In [20]:
import time

for airport in airport_staids:
    print(airport)
    
    for onemonth in monthly_ranges:
    
        querystring = {
            "station":airport_staids[airport]
            ,"start": onemonth[0] # slice the start date from the iterator
            ,"end": onemonth[1] # slice the end date from the iterator
            ,"model":"true"
        }
         
        print(querystring) 
       
        time.sleep(0.34)
    print()

LAX
{'station': 72295, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72295, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

JFK
{'station': 71845, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 71845, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

DEN
{'station': 72565, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72565, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

DTW
{'station': 72537, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72537, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

ORD
{'station': 72530, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72530, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

ATL
{'station': 72219, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72219, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}



In [21]:
#  let's catch each response in a dictionary. create an empty dictionary with the following keys:
weather_hourly_dict = {'extracted_at':[], 
                       'airport_code':[], 
                       'station_id':[], 
                       'extracted_data':[]}

# API CALL hourly (station) - for the syntax: see the rapidapi interface

url = "https://meteostat.p.rapidapi.com/stations/hourly"

headers = {
        "X-RapidAPI-Key": api_key,
        "X-RapidAPI-Host": "meteostat.p.rapidapi.com"
}


# double for-loop for the querystrings
for airport in airport_staids:
    
    # adding some logs
    print(airport) 
    
    for onemonth in monthly_ranges:
    
        querystring = {
            "station":airport_staids[airport]
            ,"start":onemonth[0]
            ,"end":onemonth[1]
            ,"model":"true"
        }
        
        # making one call with the current querystring
        response = requests.get(url, headers=headers, params=querystring)
        
        # adding some logs to catch errors
        if response.status_code != 200:
            print(f'status code {response.status_code} -> research error')
            print(querystring, end="\n\n")
        else:
            print(querystring)
        
        # appending data to the dictionary:
        weather_hourly_dict['extracted_at'].append(datetime.now())                # timestamp,
        weather_hourly_dict['airport_code'].append(airport)                       # airport code
        weather_hourly_dict['station_id'].append(airport_staids[airport])         # weater Station ID
        weather_hourly_dict['extracted_data'].append(json.loads(response.text))   # JSON string
        
        time.sleep(0.34)
        
    print()

LAX
{'station': 72295, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72295, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

JFK
{'station': 71845, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 71845, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

DEN
{'station': 72565, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72565, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

DTW
{'station': 72537, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72537, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

ORD
{'station': 72530, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72530, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}

ATL
{'station': 72219, 'start': '2022-12-01', 'end': '2022-12-31', 'model': 'true'}
{'station': 72219, 'start': '2023-01-01', 'end': '2023-01-31', 'model': 'true'}



In [22]:
weather_hourly_dict

{'extracted_at': [datetime.datetime(2025, 7, 17, 10, 30, 15, 617482),
  datetime.datetime(2025, 7, 17, 10, 30, 17, 367580),
  datetime.datetime(2025, 7, 17, 10, 30, 18, 935360),
  datetime.datetime(2025, 7, 17, 10, 30, 20, 520268),
  datetime.datetime(2025, 7, 17, 10, 30, 22, 165882),
  datetime.datetime(2025, 7, 17, 10, 30, 23, 739566),
  datetime.datetime(2025, 7, 17, 10, 30, 25, 230055),
  datetime.datetime(2025, 7, 17, 10, 30, 26, 824971),
  datetime.datetime(2025, 7, 17, 10, 30, 27, 518078),
  datetime.datetime(2025, 7, 17, 10, 30, 29, 163045),
  datetime.datetime(2025, 7, 17, 10, 30, 30, 729659),
  datetime.datetime(2025, 7, 17, 10, 30, 32, 397573)],
 'airport_code': ['LAX',
  'LAX',
  'JFK',
  'JFK',
  'DEN',
  'DEN',
  'DTW',
  'DTW',
  'ORD',
  'ORD',
  'ATL',
  'ATL'],
 'station_id': [72295,
  72295,
  71845,
  71845,
  72565,
  72565,
  72537,
  72537,
  72530,
  72530,
  72219,
  72219],
 'extracted_data': [{'meta': {'generated': '2025-07-17 08:30:15'},
   'data': [{'time':

In [23]:
weather_hourly_df = pd.DataFrame(weather_hourly_dict)
weather_hourly_df

,extracted_at,airport_code,station_id,extracted_data
0,2025-07-17 10:30:15.617482,LAX,72295,"{'meta': {'generated': '2025-07-17 08:30:15'},..."
1,2025-07-17 10:30:17.367580,LAX,72295,"{'meta': {'generated': '2025-07-17 08:30:17'},..."
2,2025-07-17 10:30:18.935360,JFK,71845,"{'meta': {'generated': '2025-07-17 08:30:18'},..."
3,2025-07-17 10:30:20.520268,JFK,71845,"{'meta': {'generated': '2025-07-17 08:30:20'},..."
4,2025-07-17 10:30:22.165882,DEN,72565,"{'meta': {'generated': '2025-07-17 08:30:22'},..."
5,2025-07-17 10:30:23.739566,DEN,72565,"{'meta': {'generated': '2025-07-17 08:30:23'},..."
6,2025-07-17 10:30:25.230055,DTW,72537,"{'meta': {'generated': '2025-07-17 08:30:25'},..."
7,2025-07-17 10:30:26.824971,DTW,72537,"{'meta': {'generated': '2025-07-17 08:30:26'},..."
8,2025-07-17 10:30:27.518078,ORD,72530,"{'meta': {'generated': '2025-07-17 08:30:27'},..."
9,2025-07-17 10:30:29.163045,ORD,72530,"{'meta': {'generated': '2025-07-17 08:30:29'},..."


In [24]:
df_JFK_jan24 = pd.json_normalize(pd.json_normalize(weather_hourly_df['extracted_data']).loc[0, 'data'])
df_JFK_jan24

,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
0,2022-12-01 00:00:00,15.0,9.4,69.0,0.0,None,220.0,13.0,None,1014.6,None,3
1,2022-12-01 01:00:00,14.4,9.4,72.0,0.0,None,210.0,9.4,None,1014.8,None,3
2,2022-12-01 02:00:00,14.4,9.0,70.0,0.0,None,210.0,7.6,None,1015.1,None,3
3,2022-12-01 03:00:00,13.9,8.9,72.0,0.0,None,190.0,7.6,None,1015.2,None,3
4,2022-12-01 04:00:00,14.4,7.7,64.0,0.0,None,150.0,9.4,None,1015.6,None,3
...,...,...,...,...,...,...,...,...,...,...,...,...
739,2022-12-31 19:00:00,15.6,14.5,93.0,NaN,None,230.0,16.6,None,1017.1,None,4
740,2022-12-31 20:00:00,15.6,14.5,93.0,NaN,None,230.0,14.8,None,1016.2,None,4
741,2022-12-31 21:00:00,15.0,14.4,96.0,NaN,None,220.0,16.6,None,1015.0,None,4
742,2022-12-31 22:00:00,15.0,13.9,93.0,NaN,None,200.0,18.4,None,1014.3,None,8


Ingesting Data in to DB

In [25]:
# getting API and DB credentials - Alternative 1: dotenv_values()

config = dotenv_values()
 
pg_user = config['POSTGRES_USER'] # align the key labels with your .env file
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

In [26]:
# updating the url
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

# creating the engine
engine = create_engine(url, echo=False)

In [27]:
# defining data types for the DB
dtype_dict = {
    'extracted_at':types.DateTime,
    'airport_code': types.String,
    'station_id': types.Integer,
    'extracted_data':postgres_json
             }

In [28]:
weather_hourly_df.to_sql(name = 'weather_hourly_raw', 
                       con = engine, 
                       schema = pg_schema, # pandas is allowing to specify, in which schema the table shall be created
                       if_exists='replace', 
                       dtype=dtype_dict,
                       index=False
                      )

12